In [1]:
# import sys
# sys.path.append('/Users/sashamorozov/Documents/Code/NCCR/transport_frames')
import transport_frames.new_modules.graphbuilder.graph as sav
import transport_frames.new_modules.framebuilder.frame as f
import transport_frames.new_modules.utils.helper_funcs as he
import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.ops import unary_union


import numpy as np
import momepy

# regions = gpd.read_file('/Users/polina/Desktop/github/transport_frames/data/lo_gdfs/regions_of_russia.geojson')
# regions = regions[regions['ISO3166-2']!='RU-CHU']

# polygon = ox.geocode_to_gdf('R81993',by_osmid=True)


/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lo_polygon = ox.geocode_to_gdf('R176095', by_osmid=True).to_crs(epsg=32636)
spb_polygon = ox.geocode_to_gdf('R337422', by_osmid=True).to_crs(epsg=32636).buffer(3000)
city = lo_polygon.union(spb_polygon).to_crs(epsg=4326) #  get lo polygon
polygon = gpd.GeoDataFrame([{'name': 'Ленинградская область', 'geometry': city[0]}], crs="EPSG:4326")

In [3]:
g = sav.Graph.from_polygon(polygon, crs=32636)

# g=sav.Graph.from_geocode('Тульская область')

# g=sav.Graph.from_geocode('R81993',by_osmid=True,crs=32636)

# tula = he.convert_geometry_from_wkt(nx.read_graphml('tula.graphml'))
# tula = he.convert_list_attr_from_str(tula)
# g = sav.Graph(tula)


# g.classify_roads()



2024-08-29 14:57:44.973 | INFO     | transport_frames.new_modules.graphbuilder.graph:from_polygon:69 - Downloading the graph from OSM...
2024-08-29 14:58:47.942 | INFO     | transport_frames.new_modules.graphbuilder.graph:_prepare_attrs:134 - Preparing the graph...
2024-08-29 15:00:36.801 | INFO     | transport_frames.new_modules.graphbuilder.graph:_prepare_attrs:171 - Graph is ready!


In [4]:
# from transport_frames.new_modules.models.graph_validation import GraphNode, GraphEdge, GraphMetadata

# for d in map(lambda e: e[2], g.graph.edges(data=True)):
#             d = GraphEdge(**d).__dict__

In [4]:
g.classify_roads()

In [18]:
n, e = momepy.nx_to_gdf(g.graph)

In [21]:
e.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 314575 entries, 0 to 314574
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   length_meter  314575 non-null  float64 
 1   geometry      314575 non-null  geometry
 2   highway       314575 non-null  object  
 3   ref           27539 non-null   object  
 4   reg           314575 non-null  int64   
 5   maxspeed      314575 non-null  float64 
 6   time_min      314575 non-null  float64 
 7   node_start    314575 non-null  int64   
 8   node_end      314575 non-null  int64   
dtypes: float64(3), geometry(1), int64(3), object(2)
memory usage: 21.6+ MB


In [28]:
momepy.nx_to_gdf(g.graph)[1]

,length_meter,geometry,highway,ref,reg,maxspeed,time_min,node_start,node_end
0,158.791,"LINESTRING (316822.012 6514139.318, 316864.213...",tertiary,NaN,3,16.666667,0.159,0,1
1,209.104,"LINESTRING (316822.012 6514139.318, 316878.621...",tertiary,NaN,3,16.666667,0.209,0,2
2,109.356,"LINESTRING (316822.012 6514139.318, 316793.074...",tertiary,NaN,3,16.666667,0.109,0,3
3,178.371,"LINESTRING (316822.012 6514139.318, 316821.070...",residential,NaN,3,16.666667,0.178,0,4
4,166.563,"LINESTRING (316864.213 6514292.399, 316905.768...",tertiary,NaN,3,16.666667,0.167,1,284
...,...,...,...,...,...,...,...,...,...
314570,42.975,"LINESTRING (222429.409 6727810.800, 222434.555...",unclassified,NaN,3,16.666667,0.043,126845,126846
314571,443.462,"LINESTRING (222453.684 6727777.907, 222478.142...",unclassified,NaN,3,16.666667,0.443,126846,126846
314572,443.462,"LINESTRING (222453.684 6727777.907, 222468.710...",unclassified,NaN,3,16.666667,0.443,126846,126846
314573,42.975,"LINESTRING (222453.684 6727777.907, 222434.555...",unclassified,NaN,3,16.666667,0.043,126846,126845


In [5]:
cities = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_district_admin_centres_18_nodes.geojson')
cities = cities[['name188', 'geometry']].rename(columns={'name188': 'name'})
# cities = cities[cities.is_admin_centre_district==1].reset_index()
# cities = cities[['name','geometry']]

In [6]:
regions = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/regions_of_russia.geojson')
regions = regions[regions['ISO3166-2']!='RU-CHU']


In [7]:
frame = f.Frame(g.graph,regions,polygon,cities)

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/new_modules/framebuilder/frame.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_nodes['exit_country'] = exits.reindex(gdf_nodes.index)['exit_country'].fillna(False)
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/new_modules/f

In [8]:
gg = frame.frame

In [9]:
node_start = next(node for node, data in gg.nodes(data=True) if data.get('nodeID') == 21729)
node_end = next(node for node, data in gg.nodes(data=True) if data.get('nodeID') == 21733)

In [10]:
shortest_path = nx.shortest_path(gg, source=node_start, target=node_end, weight="time_min")

In [16]:
shortest_path

[21729, 21731, 21732, 21733]

In [11]:
for node in shortest_path:
    gg.nodes[node]['reg_1'] = True   
    gg.nodes[node]['reg_2'] = False

In [12]:
for i in range(len(shortest_path) - 1):
    u, v = shortest_path[i], shortest_path[i + 1]

    
    if gg.has_edge(u, v):  
        for edge_key in gg[u][v]:
            gg[u][v][edge_key]['reg'] = 1  

    
    if gg.has_edge(v, u):  
        for edge_key in gg[v][u]:
            gg[v][u][edge_key]['reg'] = 1  

In [21]:
gg

In [23]:
n, e = momepy.nx_to_gdf(gg)

In [14]:
n = n.to_crs(epsg=4326)
e = e.to_crs(epsg=4326)

In [15]:
combined_gdf = gpd.GeoDataFrame(pd.concat([n, e], ignore_index=True))

In [16]:
combined_gdf.to_file('transport_frame_Leningrad_region.gpkg', driver='GPKG')

In [ ]:
def tolist(var):
    if isinstance(var,list):
        return str(var)
    return var
combined_gdf= combined_gdf.applymap(tolist)

In [22]:
e[e.reg==1].explore()

In [29]:
n.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 23160 entries, 0 to 23159
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   x             23160 non-null  float64 
 1   y             23160 non-null  float64 
 2   reg_1         23160 non-null  bool    
 3   reg_2         23160 non-null  bool    
 4   nodeID        23160 non-null  int64   
 5   exit          23160 non-null  bool    
 6   exit_country  23160 non-null  bool    
 7   weight        23160 non-null  float64 
 8   city_name     17 non-null     object  
 9   geometry      23160 non-null  geometry
dtypes: bool(4), float64(3), geometry(1), int64(1), object(1)
memory usage: 1.1+ MB


In [ ]:
# polygon - name,geo
# regions - id, geometry [poly,multipoly]
# centers - name, geometry
# country - geometry [poly,multipoly]


In [ ]:
n,e = momepy.nx_to_gdf(frame.frame)
e

In [ ]:
n

In [ ]:
# gg= he.convert_list_attr_to_str(g.graph)
# for node,e,data in gg.edges(data=True):
#     if 'geometry' in data:
#         data['geometry']=str(data['geometry'])
# nx.write_graphml(gg, f"tula.graphml")